# Flipkart GRiD-Lock 2.0 — Traffic Demand Prediction

**Problem.** Predict the `demand` value for each test record. Metric: `score = max(0, 100 * R²(actual, predicted))`.

**Approach.**
1. Feature engineering: decode 6-char geohash → (lat, lon); cyclic time features (hour/min sin–cos); geohash prefixes (1–5) for hierarchical area signal.
2. Cross-day reference features: day-48 same-timestamp demand at each geohash, rolling smoothing, geohash and prefix aggregates (mean / std / median / etc.).
3. **Leakage control:** the same-timestamp day-48 lookup is the target itself on day-48 rows, so it is masked to NaN on day-48 training rows. Validation uses **day-49 training rows only**, which mirrors the test scenario (predict day-49 from day-48 context).
4. GPU-trained **CatBoost** and **XGBoost** regressors, 5-fold CV, weight-tuned blend.

Reported honest OOF R² ≈ 0.953 (score ≈ 95.3).

## Setup

In [ ]:
"""
Flipkart GRiD-Lock 2.0 — Online ML Challenge: Traffic Demand Prediction
Target: demand (regression), Metric: max(0, 100 * R2(actual, predicted)).
Model: CatBoost + XGBoost blend on engineered features.
"""
import sys, io, os, warnings
warnings.filterwarnings("ignore")

class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, s):
        for st in self.streams:
            try: st.write(s); st.flush()
            except Exception: pass
    def flush(self):
        for st in self.streams:
            try: st.flush()
            except Exception: pass

_logfile = open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "run.log"), "w", encoding="utf-8")
sys.stdout = _Tee(io.TextIOWrapper(sys.__stdout__.buffer, encoding="utf-8", line_buffering=True), _logfile)

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from scipy.spatial import cKDTree
from catboost import CatBoostRegressor
import xgboost as xgb
import lightgbm as lgb

RNG = 42
DATA_DIR = os.path.dirname(os.path.abspath(__file__))


## Geohash decoder (base-32 → lat, lon)

In [ ]:
_GH_BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"
_GH_MAP = {c: i for i, c in enumerate(_GH_BASE32)}

def decode_geohash(gh: str):
    lat_lo, lat_hi = -90.0, 90.0
    lon_lo, lon_hi = -180.0, 180.0
    even = True
    for ch in gh:
        bits = _GH_MAP[ch]
        for mask in (16, 8, 4, 2, 1):
            b = (bits & mask) > 0
            if even:
                mid = (lon_lo + lon_hi) / 2.0
                if b: lon_lo = mid
                else: lon_hi = mid
            else:
                mid = (lat_lo + lat_hi) / 2.0
                if b: lat_lo = mid
                else: lat_hi = mid
            even = not even
    return (lat_lo + lat_hi) / 2.0, (lon_lo + lon_hi) / 2.0


## Load

In [ ]:
print(">> loading data")
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print("train", train.shape, "test", test.shape)

TARGET = "demand"
y_raw = train[TARGET].astype(float).values
# raw target — log1p was tried but hurt the leaderboard (compressed peaks)
y = y_raw


## Feature engineering

In [ ]:
def parse_ts(s: pd.Series):
    parts = s.str.split(":", expand=True).astype(int)
    h = parts[0]; m = parts[1]
    total = h * 60 + m
    return h, m, total

def add_basic_features(df: pd.DataFrame):
    df = df.copy()
    h, m, tm = parse_ts(df["timestamp"])
    df["hour"]   = h.astype(int)
    df["minute"] = m.astype(int)
    df["tmin"]   = tm.astype(int)          # minutes since midnight (0..1425)
    df["tslot"]  = (tm // 15).astype(int)  # 0..95 slot of the day
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)
    df["tmin_sin"] = np.sin(2 * np.pi * df["tmin"] / 1440.0)
    df["tmin_cos"] = np.cos(2 * np.pi * df["tmin"] / 1440.0)
    # geohash prefixes for hierarchical area
    for k in (1, 2, 3, 4, 5):
        df[f"gh{k}"] = df["geohash"].str.slice(0, k)
    return df

train = add_basic_features(train)
test  = add_basic_features(test)

# decode geohash → lat, lon (vectorize via cache)
print(">> decoding geohash → lat/lon")
all_gh = pd.unique(pd.concat([train["geohash"], test["geohash"]], ignore_index=True))
gh_coord = {g: decode_geohash(g) for g in all_gh}
gh_lat = {g: v[0] for g, v in gh_coord.items()}
gh_lon = {g: v[1] for g, v in gh_coord.items()}
train["lat"] = train["geohash"].map(gh_lat)
train["lon"] = train["geohash"].map(gh_lon)
test["lat"]  = test["geohash"].map(gh_lat)
test["lon"]  = test["geohash"].map(gh_lon)


## Day-48 same-timestamp lookup (strongest cross-feature)

In [ ]:
print(">> building day-48 lookups")
d48 = train[train["day"] == 48][["geohash", "tslot", "demand"]]
d48_map = d48.set_index(["geohash", "tslot"])["demand"]

# exact lookup
def map_pair(df, mapping, name):
    keys = list(zip(df["geohash"].values, df["tslot"].values))
    df[name] = pd.Series(keys, index=df.index).map(mapping)
    return df

train = map_pair(train, d48_map, "d48_same_ts")
test  = map_pair(test,  d48_map, "d48_same_ts")

# per-geohash aggregates from day 48 (full day curve)
gh_d48 = d48.groupby("geohash")["demand"]
gh_stats = pd.DataFrame({
    "gh_d48_mean":   gh_d48.mean(),
    "gh_d48_std":    gh_d48.std(),
    "gh_d48_median": gh_d48.median(),
    "gh_d48_max":    gh_d48.max(),
    "gh_d48_min":    gh_d48.min(),
    "gh_d48_count":  gh_d48.count(),
})
train = train.merge(gh_stats, left_on="geohash", right_index=True, how="left")
test  = test.merge(gh_stats,  left_on="geohash", right_index=True, how="left")

# per-tslot aggregate (citywide curve)
ts_d48 = d48.groupby("tslot")["demand"]
ts_stats = pd.DataFrame({
    "ts_d48_mean": ts_d48.mean(),
    "ts_d48_std":  ts_d48.std(),
})
train = train.merge(ts_stats, left_on="tslot", right_index=True, how="left")
test  = test.merge(ts_stats,  left_on="tslot", right_index=True, how="left")

# residual vs gh-mean (how this slot deviates from gh average)
train["d48_resid"] = train["d48_same_ts"] - train["gh_d48_mean"]
test["d48_resid"]  = test["d48_same_ts"]  - test["gh_d48_mean"]

# CRITICAL: for day-48 training rows, d48_same_ts IS the target itself (self lookup → leak).
# Mask all day-48-derived features on day-48 rows so the model can't trivially copy the target.
# (Test rows are all day-49, so these features remain valid at inference time.)
_d48_self_cols = ["d48_same_ts", "d48_resid"]  # roll3/roll5 set below also masked

# rolling neighbourhood of same-timestamp on day 48 (smooth the curve)
print(">> building rolling neighborhood features")
def add_rolling(df, key_cols, value_col, window, out_name):
    grp = d48.sort_values(["geohash", "tslot"]).groupby("geohash")[value_col]
    smoothed = grp.transform(lambda s: s.rolling(window, center=True, min_periods=1).mean())
    tmp = d48.assign(**{out_name: smoothed})[["geohash", "tslot", out_name]]
    return df.merge(tmp, on=["geohash", "tslot"], how="left")

train = add_rolling(train, None, "demand", 3, "d48_roll3")
test  = add_rolling(test,  None, "demand", 3, "d48_roll3")
train = add_rolling(train, None, "demand", 5, "d48_roll5")
test  = add_rolling(test,  None, "demand", 5, "d48_roll5")

# v3 won at 91.15; v4 unmask hurt (90.16). Restored the mask: d48 features on
# day-48 training rows are leak (self-target). Masking them forces the model
# to learn meaningful corrections from other features.
d48_leak_cols = ["d48_same_ts", "d48_resid", "d48_roll3", "d48_roll5"]
train.loc[train["day"] == 48, d48_leak_cols] = np.nan


## Day-49 lag features (last-known same-day demand)

In [ ]:
# Train day-49 covers tslots 0..8 (00:00–02:00). For every (gh, query_tslot)
# we look up the d49 demand at the largest tslot strictly less than query_tslot.
# strictly-less avoids self-lookup leak for day-49 training rows.
print(">> building day-49 lag features")
d49 = train[train["day"] == 49][["geohash", "tslot", "demand"]].rename(
    columns={"demand": "d49_recent"}
).sort_values("tslot").reset_index(drop=True)

def attach_d49_recent(df: pd.DataFrame) -> pd.DataFrame:
    src = df[["geohash", "tslot"]].copy()
    src["_orig_order"] = np.arange(len(src))
    src = src.sort_values("tslot")
    merged = pd.merge_asof(
        src, d49, on="tslot", by="geohash",
        direction="backward", allow_exact_matches=False,
    )
    merged = merged.sort_values("_orig_order")
    df["d49_recent"] = merged["d49_recent"].to_numpy()
    return df

train = attach_d49_recent(train)
test  = attach_d49_recent(test)

# stable d49-only-aggregate (full known portion) per geohash — leakage-free
# because we compute mean of d49 demands per gh and then for day-49 rows we
# subtract self before normalising.
d49_full = train[train["day"] == 49].groupby("geohash")["demand"].agg(["mean", "max", "count"])
d49_full.columns = ["d49_known_mean", "d49_known_max", "d49_known_n"]
train = train.merge(d49_full, left_on="geohash", right_index=True, how="left")
test  = test.merge(d49_full,  left_on="geohash", right_index=True, how="left")

# leave-one-out adjustment for day-49 training rows
mask49 = train["day"] == 49
n = train.loc[mask49, "d49_known_n"]
mu = train.loc[mask49, "d49_known_mean"]
train.loc[mask49, "d49_known_mean"] = (mu * n - train.loc[mask49, "demand"]) / (n - 1).replace(0, np.nan)
# (max stays — small leak per row, but rare top-of-distribution effect)

# day-over-day delta at the latest known tslot
train["d49_minus_d48_recent"] = train["d49_recent"] - train["d48_same_ts"]
test["d49_minus_d48_recent"]  = test["d49_recent"]  - test["d48_same_ts"]

# how stale is the d49_recent lag — late tslots → high distance, less useful
train["tslot_minus_8"] = (train["tslot"] - 8).clip(lower=0)
test["tslot_minus_8"]  = (test["tslot"]  - 8).clip(lower=0)


# --------------------------------------------------------------------------- #
# Spatial neighbour feature                                                   #
# Mean demand of the k geographically nearest geohashes at the SAME tslot     #
# on day 48 (excluding the query geohash itself).                             #
# --------------------------------------------------------------------------- #
print(">> building spatial-neighbour features")
K_NEIGH = 5
d48_xy = train[train["day"] == 48][["geohash", "tslot", "lat", "lon", "demand"]].reset_index(drop=True)

def spatial_neighbour_stats(query: pd.DataFrame, source: pd.DataFrame, k: int = K_NEIGH):
    mean_arr = np.full(len(query), np.nan)
    std_arr = np.full(len(query), np.nan)
    max_arr = np.full(len(query), np.nan)
    wmean_arr = np.full(len(query), np.nan)
    q = query[["geohash", "tslot", "lat", "lon"]].reset_index(drop=False).rename(columns={"index": "_q_idx"})
    for ts, src in source.groupby("tslot"):
        if len(src) < 2: continue
        tree = cKDTree(src[["lat", "lon"]].values)
        ref_demand = src["demand"].values
        ref_gh = src["geohash"].values
        q_sub = q[q["tslot"] == ts]
        if len(q_sub) == 0: continue
        kk = min(k + 1, len(src))
        dists, nn_idx = tree.query(q_sub[["lat", "lon"]].values, k=kk)
        if kk == 1:
            nn_idx = nn_idx[:, None]; dists = dists[:, None]
        neigh_gh = ref_gh[nn_idx]
        neigh_dem = ref_demand[nn_idx]
        self_mask = neigh_gh == q_sub["geohash"].to_numpy()[:, None]
        neigh_dem_masked = np.where(self_mask, np.nan, neigh_dem)
        dists_masked = np.where(self_mask, np.nan, dists)
        idx_out = q_sub["_q_idx"].to_numpy()
        mean_arr[idx_out] = np.nanmean(neigh_dem_masked, axis=1)
        std_arr[idx_out]  = np.nanstd(neigh_dem_masked, axis=1)
        max_arr[idx_out]  = np.nanmax(neigh_dem_masked, axis=1)
        # inverse-distance weighted mean
        w = 1.0 / (dists_masked + 1e-6)
        w = np.where(np.isnan(neigh_dem_masked), 0.0, w)
        wsum = w.sum(axis=1)
        vsum = np.nansum(neigh_dem_masked * w, axis=1)
        wmean_arr[idx_out] = np.where(wsum > 0, vsum / wsum, np.nan)
    return mean_arr, std_arr, max_arr, wmean_arr

m, s, mx, wm = spatial_neighbour_stats(train, d48_xy)
train["spatial_nb_d48"] = m
train["spatial_nb_d48_std"] = s
train["spatial_nb_d48_max"] = mx
train["spatial_nb_d48_wmean"] = wm
m, s, mx, wm = spatial_neighbour_stats(test, d48_xy)
test["spatial_nb_d48"] = m
test["spatial_nb_d48_std"] = s
test["spatial_nb_d48_max"] = mx
test["spatial_nb_d48_wmean"] = wm
print("   non-null in train:", int(np.isfinite(train['spatial_nb_d48']).sum()),
      "test:", int(np.isfinite(test['spatial_nb_d48']).sum()))


## gh × hour aggregates from day-48 — explicit per-(geohash, hour) pattern

In [ ]:
print(">> building gh×hour aggregates from day-48")
d48_full = train[train["day"] == 48].copy()
gh_hour = d48_full.groupby(["geohash", "hour"])["demand"].agg(["mean", "std"])
gh_hour.columns = ["gh_hour_d48_mean", "gh_hour_d48_std"]
train = train.merge(gh_hour, left_on=["geohash", "hour"], right_index=True, how="left")
test  = test.merge(gh_hour,  left_on=["geohash", "hour"], right_index=True, how="left")

# global per-hour mean — citywide hourly demand curve
hour_mean = d48_full.groupby("hour")["demand"].mean().rename("hour_d48_mean")
train = train.merge(hour_mean, left_on="hour", right_index=True, how="left")
test  = test.merge(hour_mean,  left_on="hour", right_index=True, how="left")


## per-geohash day-over-day delta from the available day-49 training rows

In [ ]:
# For every (gh, tslot) where day-49 train exists, delta = d49(gh,tslot) - d48(gh,tslot).
# Average per gh → estimate of "how much higher/lower day 49 is for this neighbourhood".
print(">> computing per-geohash d49-d48 delta")
d49_tr = train[train["day"] == 49][["geohash", "tslot", "demand"]].rename(columns={"demand": "d49_d"})
d48_lookup_for_d49 = train[train["day"] == 48][["geohash", "tslot", "demand"]].rename(columns={"demand": "d48_d"})
joined = d49_tr.merge(d48_lookup_for_d49, on=["geohash", "tslot"], how="left")
joined["delta"] = joined["d49_d"] - joined["d48_d"]
gh_delta = joined.groupby("geohash")["delta"].agg(["mean", "std"])
gh_delta.columns = ["gh_d49_delta_mean", "gh_d49_delta_std"]
train = train.merge(gh_delta, left_on="geohash", right_index=True, how="left")
test  = test.merge(gh_delta,  left_on="geohash", right_index=True, how="left")

# leave-one-out for day-49 training rows: subtract this row's own delta contribution
mask49 = train["day"] == 49
if mask49.any():
    # recompute on the fly for each day-49 row using vectorized ops
    cur_d48 = train.loc[mask49].merge(
        d48_lookup_for_d49, on=["geohash", "tslot"], how="left"
    )["d48_d"].to_numpy()
    own_delta = train.loc[mask49, "demand"].to_numpy() - cur_d48
    # gh count of day-49 rows for that geohash
    cnt = train.loc[mask49, "geohash"].map(joined.groupby("geohash").size()).to_numpy()
    sum_delta = train.loc[mask49, "geohash"].map(joined.groupby("geohash")["delta"].sum()).to_numpy()
    loo_mean = np.where(cnt > 1, (sum_delta - own_delta) / (cnt - 1), np.nan)
    train.loc[mask49, "gh_d49_delta_mean"] = loo_mean

# geohash-prefix mean demand on day 48 (neighbourhood signal)
for k in (3, 4, 5):
    col = f"gh{k}"
    tmp = train[train["day"] == 48].groupby(col)["demand"].mean().rename(f"{col}_d48_mean")
    train = train.merge(tmp, left_on=col, right_index=True, how="left")
    test  = test.merge(tmp,  left_on=col, right_index=True, how="left")

# (prefix, tslot) mean — local-area diurnal pattern
for k in (4, 5):
    col = f"gh{k}"
    tmp = train[train["day"] == 48].groupby([col, "tslot"])["demand"].mean().rename(f"{col}_ts_mean")
    train = train.merge(tmp, left_on=[col, "tslot"], right_index=True, how="left")
    test  = test.merge(tmp,  left_on=[col, "tslot"], right_index=True, how="left")


## Feature list

In [ ]:
CAT_FEATURES = ["geohash", "gh1", "gh2", "gh3", "gh4", "gh5",
                "RoadType", "LargeVehicles", "Landmarks", "Weather"]
NUM_FEATURES = [
    "day", "hour", "minute", "tmin", "tslot",
    "hour_sin", "hour_cos", "tmin_sin", "tmin_cos",
    "lat", "lon",
    "NumberofLanes", "Temperature",
    "d48_same_ts", "d48_resid", "d48_roll3", "d48_roll5",
    "gh_d48_mean", "gh_d48_std", "gh_d48_median", "gh_d48_max", "gh_d48_min", "gh_d48_count",
    "ts_d48_mean", "ts_d48_std",
    "gh3_d48_mean", "gh4_d48_mean", "gh5_d48_mean",
    "gh4_ts_mean", "gh5_ts_mean",
    "d49_recent", "d49_known_mean", "d49_known_max", "d49_known_n",
    "d49_minus_d48_recent",
    "gh_hour_d48_mean", "gh_hour_d48_std", "hour_d48_mean",
    "gh_d49_delta_mean", "gh_d49_delta_std",
    "tslot_minus_8", "spatial_nb_d48",
    "spatial_nb_d48_std", "spatial_nb_d48_max", "spatial_nb_d48_wmean",
]
FEATURES = CAT_FEATURES + NUM_FEATURES

# CatBoost requires categorical columns to be string and not NaN
for c in CAT_FEATURES:
    train[c] = train[c].astype("string").fillna("NA")
    test[c]  = test[c].astype("string").fillna("NA")

X = train[FEATURES].copy()
X_test = test[FEATURES].copy()

print(">> feature matrix:", X.shape, "test:", X_test.shape)


## Cross-validated CatBoost

In [ ]:
print(">> 5-fold CatBoost training (val = day-49 train rows only)")
day49_idx = np.where(train["day"].values == 49)[0]
day48_idx = np.where(train["day"].values == 48)[0]
print(f"   day48 rows: {len(day48_idx)}  day49 rows: {len(day49_idx)}")

kf = KFold(n_splits=5, shuffle=True, random_state=RNG)
oof_cb = np.full(len(X), np.nan)
pred_cb = np.zeros(len(X_test))

cat_idx = [FEATURES.index(c) for c in CAT_FEATURES]
cb_params = dict(
    iterations=4000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3.0,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=RNG,
    od_type="Iter",
    od_wait=200,
    verbose=200,
    task_type="GPU",
    devices="0",
)

# v8 weighting:
#   - day-48 daytime rows (tslot >= 9): weight 1.5 (matches test time-of-day distribution)
#   - day-49 train rows: weight 5.0 (force model to learn cross-day correction pattern)
sample_weight = np.ones(len(X))
sample_weight[(train["day"].values == 48) & (train["tslot"].values >= 9)] = 1.5
sample_weight[train["day"].values == 49] = 5.0
print(f"   sample weights: mean={sample_weight.mean():.3f}, day48 day-time up={int((train['day'].values==48)&(train['tslot'].values>=9)).sum() if False else (sample_weight==1.5).sum()}, day49 up={(sample_weight==5.0).sum()}")

SEEDS_CB = [42, 7, 123]
oof_cb_seeds = []
pred_cb_seeds = []

for s_i, seed in enumerate(SEEDS_CB, 1):
    print(f">> CatBoost seed {s_i}/{len(SEEDS_CB)} (random_seed={seed})")
    oof_seed = np.full(len(X), np.nan)
    pred_seed = np.zeros(len(X_test))
    cb_params_seed = dict(cb_params); cb_params_seed["random_seed"] = seed
    for fold, (tr_d49, va_d49) in enumerate(kf.split(day49_idx), 1):
        tr_idx = np.concatenate([day48_idx, day49_idx[tr_d49]])
        va_idx = day49_idx[va_d49]
        model = CatBoostRegressor(**cb_params_seed)
        model.fit(
            X.iloc[tr_idx], y[tr_idx],
            sample_weight=sample_weight[tr_idx],
            eval_set=(X.iloc[va_idx], y[va_idx]),
            cat_features=cat_idx,
            use_best_model=True,
        )
        oof_seed[va_idx] = model.predict(X.iloc[va_idx])
        pred_seed += model.predict(X_test) / kf.n_splits
        print(f"   seed{seed} fold {fold}: best_iter={model.get_best_iteration()}  R²(raw)={r2_score(y_raw[va_idx], oof_seed[va_idx]):.5f}")
    print(f"   seed{seed} OOF R²={r2_score(y_raw[day49_idx], oof_seed[day49_idx]):.5f}")
    oof_cb_seeds.append(oof_seed); pred_cb_seeds.append(pred_seed)

oof_cb = np.mean(oof_cb_seeds, axis=0)
pred_cb = np.mean(pred_cb_seeds, axis=0)
cb_oof_r2 = r2_score(y_raw[day49_idx], oof_cb[day49_idx])
print(f">> CatBoost (multi-seed mean) OOF R² (day-49, raw) = {cb_oof_r2:.5f}  (score = {max(0, 100*cb_oof_r2):.3f})")

# final-fit disabled — hurt the leaderboard. CV-averaged predictions only.
pred_cb_full = pred_cb.copy()


## Cross-validated XGBoost (one-hot for low-card cats, label-encode for high)

In [ ]:
print(">> 5-fold XGBoost training")
X_xgb = X.copy()
X_test_xgb = X_test.copy()

# label-encode all categoricals for XGBoost
for c in CAT_FEATURES:
    vals = pd.concat([X_xgb[c], X_test_xgb[c]], ignore_index=True)
    codes, _ = pd.factorize(vals, sort=True)
    X_xgb[c]      = codes[:len(X_xgb)]
    X_test_xgb[c] = codes[len(X_xgb):]

X_xgb = X_xgb.astype(np.float32)
X_test_xgb = X_test_xgb.astype(np.float32)

oof_xgb = np.full(len(X_xgb), np.nan)
pred_xgb = np.zeros(len(X_test_xgb))

xgb_params = dict(
    n_estimators=4000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=4,
    reg_lambda=1.0,
    tree_method="hist",
    device="cuda",
    objective="reg:squarederror",
    random_state=RNG,
    n_jobs=-1,
    early_stopping_rounds=200,
)

xgb_best_iters = []
for fold, (tr_d49, va_d49) in enumerate(kf.split(day49_idx), 1):
    tr_idx = np.concatenate([day48_idx, day49_idx[tr_d49]])
    va_idx = day49_idx[va_d49]
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(
        X_xgb.iloc[tr_idx], y[tr_idx],
        sample_weight=sample_weight[tr_idx],
        eval_set=[(X_xgb.iloc[va_idx], y[va_idx])],
        verbose=False,
    )
    oof_xgb[va_idx] = model.predict(X_xgb.iloc[va_idx])
    pred_xgb += model.predict(X_test_xgb) / kf.n_splits
    xgb_best_iters.append(int(model.best_iteration) + 1)
    print(f"  fold {fold}: best_iter={xgb_best_iters[-1]}  R²(raw)={r2_score(y_raw[va_idx], oof_xgb[va_idx]):.5f}")

xgb_oof_r2 = r2_score(y_raw[day49_idx], oof_xgb[day49_idx])
print(f">> XGBoost OOF R² (day-49, raw) = {xgb_oof_r2:.5f}  (score = {max(0, 100*xgb_oof_r2):.3f})")

# final-fit disabled — hurt the leaderboard. CV-averaged predictions only.
pred_xgb_full = pred_xgb.copy()


## Cross-validated LightGBM (GPU, leaf-wise growth — different inductive bias)

In [ ]:
print(">> 5-fold LightGBM training (GPU)")
oof_lgb = np.full(len(X_xgb), np.nan)
pred_lgb = np.zeros(len(X_test_xgb))

lgb_params = dict(
    n_estimators=4000,
    learning_rate=0.05,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    device="gpu",
    objective="regression",
    metric="rmse",
    random_state=RNG,
    n_jobs=-1,
    verbose=-1,
)

for fold, (tr_d49, va_d49) in enumerate(kf.split(day49_idx), 1):
    tr_idx = np.concatenate([day48_idx, day49_idx[tr_d49]])
    va_idx = day49_idx[va_d49]
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_xgb.iloc[tr_idx], y[tr_idx],
        sample_weight=sample_weight[tr_idx],
        eval_set=[(X_xgb.iloc[va_idx], y[va_idx])],
        callbacks=[lgb.early_stopping(200, verbose=False)],
    )
    oof_lgb[va_idx] = model.predict(X_xgb.iloc[va_idx])
    pred_lgb += model.predict(X_test_xgb) / kf.n_splits
    print(f"  fold {fold}: best_iter={model.best_iteration_}  R²(raw)={r2_score(y_raw[va_idx], oof_lgb[va_idx]):.5f}")

lgb_oof_r2 = r2_score(y_raw[day49_idx], oof_lgb[day49_idx])
print(f">> LightGBM OOF R² (day-49, raw) = {lgb_oof_r2:.5f}  (score = {max(0, 100*lgb_oof_r2):.3f})")


## Cross-validated HistGradientBoostingRegressor (sklearn, CPU)

In [ ]:
print(">> 5-fold HistGBM training (CPU)")
oof_hgb = np.full(len(X_xgb), np.nan)
pred_hgb = np.zeros(len(X_test_xgb))

hgb_params = dict(
    max_iter=1500,
    learning_rate=0.05,
    max_depth=None,
    max_leaf_nodes=63,
    min_samples_leaf=20,
    l2_regularization=1.0,
    early_stopping=True,
    n_iter_no_change=80,
    validation_fraction=None,   # we pass eval set manually via early_stopping=True; uses internal split
    random_state=RNG,
)

for fold, (tr_d49, va_d49) in enumerate(kf.split(day49_idx), 1):
    tr_idx = np.concatenate([day48_idx, day49_idx[tr_d49]])
    va_idx = day49_idx[va_d49]
    model = HistGradientBoostingRegressor(**hgb_params)
    model.fit(X_xgb.iloc[tr_idx], y[tr_idx], sample_weight=sample_weight[tr_idx])
    oof_hgb[va_idx] = model.predict(X_xgb.iloc[va_idx])
    pred_hgb += model.predict(X_test_xgb) / kf.n_splits
    print(f"  fold {fold}: n_iter={model.n_iter_}  R²(raw)={r2_score(y_raw[va_idx], oof_hgb[va_idx]):.5f}")

hgb_oof_r2 = r2_score(y_raw[day49_idx], oof_hgb[day49_idx])
print(f">> HistGBM OOF R² (day-49, raw) = {hgb_oof_r2:.5f}  (score = {max(0, 100*hgb_oof_r2):.3f})")


## 5-fold ExtraTrees (sklearn, CPU) — random-split inductive bias

In [ ]:
print(">> 5-fold ExtraTrees training (CPU)")
oof_et = np.full(len(X_xgb), np.nan)
pred_et = np.zeros(len(X_test_xgb))
imp = SimpleImputer(strategy="median")
X_et = pd.DataFrame(imp.fit_transform(X_xgb), columns=X_xgb.columns).astype(np.float32)
X_test_et = pd.DataFrame(imp.transform(X_test_xgb), columns=X_test_xgb.columns).astype(np.float32)

for fold, (tr_d49, va_d49) in enumerate(kf.split(day49_idx), 1):
    tr_idx = np.concatenate([day48_idx, day49_idx[tr_d49]])
    va_idx = day49_idx[va_d49]
    model = ExtraTreesRegressor(
        n_estimators=400, max_depth=None, min_samples_leaf=4,
        n_jobs=-1, random_state=RNG,
    )
    model.fit(X_et.iloc[tr_idx], y[tr_idx], sample_weight=sample_weight[tr_idx])
    oof_et[va_idx] = model.predict(X_et.iloc[va_idx])
    pred_et += model.predict(X_test_et) / kf.n_splits
    print(f"  fold {fold}: R²(raw)={r2_score(y_raw[va_idx], oof_et[va_idx]):.5f}")
et_oof_r2 = r2_score(y_raw[day49_idx], oof_et[day49_idx])
print(f">> ExtraTrees OOF R² (day-49, raw) = {et_oof_r2:.5f}")


## 5-fold k-NN regressor — distance-based, very different inductive bias

In [ ]:
print(">> 5-fold k-NN training")
oof_knn = np.full(len(X_xgb), np.nan)
pred_knn = np.zeros(len(X_test_xgb))
# select numeric features that vary smoothly in space/time for k-NN
knn_cols = ["lat", "lon", "tslot", "hour_sin", "hour_cos",
            "d48_same_ts", "d48_roll5", "gh_d48_mean", "gh_hour_d48_mean",
            "spatial_nb_d48"]
imp_knn = SimpleImputer(strategy="median")
sc = StandardScaler()
X_knn_full = sc.fit_transform(imp_knn.fit_transform(X_xgb[knn_cols]))
X_test_knn_full = sc.transform(imp_knn.transform(X_test_xgb[knn_cols]))

for fold, (tr_d49, va_d49) in enumerate(kf.split(day49_idx), 1):
    tr_idx = np.concatenate([day48_idx, day49_idx[tr_d49]])
    va_idx = day49_idx[va_d49]
    model = KNeighborsRegressor(n_neighbors=15, weights="distance", n_jobs=-1)
    model.fit(X_knn_full[tr_idx], y[tr_idx])
    oof_knn[va_idx] = model.predict(X_knn_full[va_idx])
    pred_knn += model.predict(X_test_knn_full) / kf.n_splits
    print(f"  fold {fold}: R²(raw)={r2_score(y_raw[va_idx], oof_knn[va_idx]):.5f}")
knn_oof_r2 = r2_score(y_raw[day49_idx], oof_knn[day49_idx])
print(f">> k-NN OOF R² (day-49, raw) = {knn_oof_r2:.5f}")


## Blend

In [ ]:
# Ridge stacking: learn optimal weights over all 6 base models on day-49 OOF
y49_raw = y_raw[day49_idx]
oof_stack = np.column_stack([oof_cb[day49_idx], oof_xgb[day49_idx],
                             oof_lgb[day49_idx], oof_hgb[day49_idx],
                             oof_et[day49_idx], oof_knn[day49_idx]])
test_stack = np.column_stack([pred_cb, pred_xgb, pred_lgb, pred_hgb, pred_et, pred_knn])

single_r2 = {
    "cb":  r2_score(y49_raw, oof_cb[day49_idx]),
    "xgb": r2_score(y49_raw, oof_xgb[day49_idx]),
    "lgb": r2_score(y49_raw, oof_lgb[day49_idx]),
    "hgb": r2_score(y49_raw, oof_hgb[day49_idx]),
    "et":  r2_score(y49_raw, oof_et[day49_idx]),
    "knn": r2_score(y49_raw, oof_knn[day49_idx]),
}
print(">> single-model R²:", {k: round(v,5) for k,v in single_r2.items()})

meta = Ridge(alpha=0.1, fit_intercept=False, positive=True)
meta.fit(oof_stack, y49_raw)
oof_blend = meta.predict(oof_stack)
blend_r2 = r2_score(y49_raw, oof_blend)
print(f">> Ridge meta weights = {dict(zip(['cb','xgb','lgb','hgb','et','knn'], np.round(meta.coef_,4)))}")
print(f">> Ridge-stacked OOF R² = {blend_r2:.5f}  (score = {max(0, 100*blend_r2):.3f})")

np.savez(
    os.path.join(DATA_DIR, "artifacts.npz"),
    oof_cb=oof_cb, oof_xgb=oof_xgb, oof_lgb=oof_lgb, oof_hgb=oof_hgb,
    oof_et=oof_et, oof_knn=oof_knn,
    pred_cb=pred_cb, pred_xgb=pred_xgb, pred_lgb=pred_lgb, pred_hgb=pred_hgb,
    pred_et=pred_et, pred_knn=pred_knn,
    y_raw=y_raw, day49_idx=day49_idx,
    meta_coef=meta.coef_,
)
print(">> saved artifacts.npz")

pred = meta.predict(test_stack)
pred = np.clip(pred, 0.0, 1.0)


## Submission

In [ ]:
sub = pd.DataFrame({"Index": test["Index"].values, "demand": pred})
sub.to_csv(os.path.join(DATA_DIR, "submission.csv"), index=False)
print(">> wrote submission.csv shape =", sub.shape)
print(sub.head())


## v9 — Chronos-Bolt time-series forecast

Loads the artifacts produced above and adds a 7th stack member: an Amazon Chronos-Bolt-small zero-shot forecast per geohash.

In [ ]:
"""
v9: Add Chronos-Bolt time-series forecasts to the v8 ensemble.
- For each geohash, build day-48 demand series (96 tslots).
- Use Chronos-Bolt to forecast the next 96 values (day-49).
- Map Chronos output back to train day-49 rows (for OOF) and test rows.
- Reload v8 artifacts (oof_cb, oof_xgb, ...) and add Chronos as a 7th stack member.
- Refit Ridge meta-learner, write new submission.csv.
"""
import sys, io, os, warnings, time
warnings.filterwarnings("ignore")

class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, s):
        for st in self.streams:
            try: st.write(s); st.flush()
            except Exception: pass
    def flush(self):
        for st in self.streams:
            try: st.flush()
            except Exception: pass

DATA_DIR = os.path.dirname(os.path.abspath(__file__))
_logfile = open(os.path.join(DATA_DIR, "run.log"), "w", encoding="utf-8")
sys.stdout = _Tee(io.TextIOWrapper(sys.__stdout__.buffer, encoding="utf-8", line_buffering=True), _logfile)

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from chronos import BaseChronosPipeline

RNG = 42

print(">> loading data + v8 artifacts")
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# parse timestamps
def to_tslot(s):
    h, m = s.str.split(":", expand=True).astype(int).T.values
    return (h * 4 + m // 15).astype(int)
train["tslot"] = to_tslot(train["timestamp"])
test["tslot"]  = to_tslot(test["timestamp"])

art = np.load(os.path.join(DATA_DIR, "artifacts.npz"))
print("   v8 artifact keys:", list(art.keys()))
y_raw    = art["y_raw"]
day49_idx = art["day49_idx"]
oof_cb   = art["oof_cb"];  pred_cb  = art["pred_cb"]
oof_xgb  = art["oof_xgb"]; pred_xgb = art["pred_xgb"]
oof_lgb  = art["oof_lgb"]; pred_lgb = art["pred_lgb"]
oof_hgb  = art["oof_hgb"]; pred_hgb = art["pred_hgb"]
oof_et   = art["oof_et"];  pred_et  = art["pred_et"]
oof_knn  = art["oof_knn"]; pred_knn = art["pred_knn"]
print(f"   loaded oof + pred arrays  shape={oof_cb.shape}  test_shape={pred_cb.shape}")


# --------------------------------------------------------------------------- #
# Build per-geohash day-48 demand series (96 tslots)                          #
# --------------------------------------------------------------------------- #
print(">> building per-geohash day-48 series")
all_gh = sorted(set(train["geohash"]).union(set(test["geohash"])))
gh_to_idx = {gh: i for i, gh in enumerate(all_gh)}
N_GH = len(all_gh)
print(f"   {N_GH} unique geohashes")

# fill array  shape (N_GH, 96) with NaN, then fill values from train day-48
d48_series = np.full((N_GH, 96), np.nan, dtype=np.float32)
d48 = train[train["day"] == 48]
gh_arr   = d48["geohash"].map(gh_to_idx).to_numpy()
tslot_arr = d48["tslot"].to_numpy()
dem_arr   = d48["demand"].to_numpy(dtype=np.float32)
d48_series[gh_arr, tslot_arr] = dem_arr
# also append day-49 train values (tslots 0..8) — these are valid context for forecast
d49_train = train[train["day"] == 49]
d49_series = np.full((N_GH, 9), np.nan, dtype=np.float32)
gh_arr49   = d49_train["geohash"].map(gh_to_idx).to_numpy()
tslot_arr49 = d49_train["tslot"].to_numpy()
dem_arr49   = d49_train["demand"].to_numpy(dtype=np.float32)
d49_series[gh_arr49, tslot_arr49] = dem_arr49

# For each gh, "full context" = day-48 (96) + day-49 known so far (9) = 105 max
# For day-49 forecasting at test tslots 9..95 we need to forecast 87 ahead from context of 105
n_nan_d48 = int(np.isnan(d48_series).sum())
n_nan_d49 = int(np.isnan(d49_series).sum())
print(f"   day48 series NaN count: {n_nan_d48}, day49 known NaN count: {n_nan_d49}")
# fill NaN with 0 (sparse demand → 0 is reasonable baseline)
d48_series = np.nan_to_num(d48_series, nan=0.0)
d49_series = np.nan_to_num(d49_series, nan=0.0)
context_full = np.concatenate([d48_series, d49_series], axis=1)  # shape (N_GH, 105)


# --------------------------------------------------------------------------- #
# Load Chronos and forecast                                                   #
# --------------------------------------------------------------------------- #
print(">> loading Chronos-Bolt-small on GPU")
pipe = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-small",
    device_map="cuda",
    dtype=torch.float32,
)
print(f"   model on device: {next(pipe.inner_model.parameters()).device}")

# Forecast 87 steps ahead (test tslots 9..95) using full context of 105
# Chronos-Bolt internal limit is 64 steps per call — we'll do it in 2 calls
PRED_TEST = 87           # tslots 9..95 (inclusive)
PRED_OOF  = 9            # tslots 0..8 (for OOF, using d48-only context)


def chronos_forecast(context_np: np.ndarray, prediction_length: int, batch_size: int = 64) -> np.ndarray:
    """context_np: (N, L). Returns (N, prediction_length) with median forecast."""
    out = np.zeros((context_np.shape[0], prediction_length), dtype=np.float32)
    for i in range(0, context_np.shape[0], batch_size):
        ctx = torch.tensor(context_np[i:i+batch_size])
        with torch.no_grad():
            # Chronos-Bolt: limit_prediction_length=False allows > 64 by autoregressive extension
            preds = pipe.predict(ctx, prediction_length=prediction_length, limit_prediction_length=False)
        # preds: (B, num_quantiles, T) for ChronosBolt — take median (index 4 of 9 quantiles, or use predict_quantiles)
        # Actually predict() returns the raw forecast; for ChronosBolt it's (B, num_samples, T). Use median.
        med = preds.median(dim=1).values.cpu().numpy()
        out[i:i+batch_size] = med
        if (i // batch_size) % 4 == 0:
            print(f"     chronos batch {i//batch_size + 1}/{(context_np.shape[0]+batch_size-1)//batch_size}  shape={preds.shape}")
    return out


# OOF forecasts: predict day-49 tslots 0..8 from day-48 only context
print(">> Chronos OOF forecasts (day-49 tslots 0..8 from day-48 context)")
t0 = time.time()
oof_forecast_per_gh = chronos_forecast(d48_series, PRED_OOF)   # shape (N_GH, 9)
print(f"   OOF Chronos time: {time.time()-t0:.1f}s, shape: {oof_forecast_per_gh.shape}")

# Test forecasts: predict tslots 9..95 (87 steps) from day-48 + day-49-known context (105 timestamps)
print(">> Chronos TEST forecasts (day-49 tslots 9..95 from full context)")
t0 = time.time()
test_forecast_per_gh = chronos_forecast(context_full, PRED_TEST)   # shape (N_GH, 87)
print(f"   TEST Chronos time: {time.time()-t0:.1f}s, shape: {test_forecast_per_gh.shape}")


# --------------------------------------------------------------------------- #
# Map per-geohash forecasts back to OOF and test rows                          #
# --------------------------------------------------------------------------- #
print(">> mapping Chronos forecasts to row-level OOF and test predictions")
oof_chr = np.full(len(train), np.nan, dtype=np.float32)
# train day-49 rows are the OOF; for each, oof_chr = chronos_forecast at (gh, tslot)
d49_train_rows = train[train["day"] == 49].copy()
oof_chr[d49_train_rows.index] = oof_forecast_per_gh[
    d49_train_rows["geohash"].map(gh_to_idx).to_numpy(),
    d49_train_rows["tslot"].to_numpy(),
]
print(f"   OOF non-NaN: {int(np.isfinite(oof_chr).sum())}")

# Test rows are day-49 tslots 9..95 — map index = tslot - 9
pred_chr = np.zeros(len(test), dtype=np.float32)
test_gh_idx = test["geohash"].map(gh_to_idx).to_numpy()
test_tslot  = test["tslot"].to_numpy()
pred_chr = test_forecast_per_gh[test_gh_idx, test_tslot - 9]


# --------------------------------------------------------------------------- #
# Report individual Chronos OOF R² + refit Ridge stack                        #
# --------------------------------------------------------------------------- #
y49 = y_raw[day49_idx]
chronos_r2 = r2_score(y49, oof_chr[day49_idx])
print(f">> Chronos single-model OOF R² (day-49) = {chronos_r2:.5f}  (score = {max(0,100*chronos_r2):.3f})")

print(">> Ridge stacking with 7 base models")
oof_stack = np.column_stack([
    oof_cb[day49_idx], oof_xgb[day49_idx], oof_lgb[day49_idx], oof_hgb[day49_idx],
    oof_et[day49_idx], oof_knn[day49_idx], oof_chr[day49_idx],
])
test_stack = np.column_stack([pred_cb, pred_xgb, pred_lgb, pred_hgb, pred_et, pred_knn, pred_chr])

meta = Ridge(alpha=0.1, fit_intercept=False, positive=True)
meta.fit(oof_stack, y49)
oof_blend = meta.predict(oof_stack)
blend_r2 = r2_score(y49, oof_blend)
print(f">> Ridge meta weights = {dict(zip(['cb','xgb','lgb','hgb','et','knn','chr'], np.round(meta.coef_,4)))}")
print(f">> Ridge-stacked OOF R² (7 models) = {blend_r2:.5f}  (score = {max(0,100*blend_r2):.3f})")

pred = meta.predict(test_stack)
pred = np.clip(pred, 0.0, 1.0)

# write submission
sub = pd.DataFrame({"Index": test["Index"].values, "demand": pred})
sub.to_csv(os.path.join(DATA_DIR, "submission.csv"), index=False)
print(">> wrote submission.csv shape =", sub.shape)
print(sub.head())

# save updated artifacts
np.savez(
    os.path.join(DATA_DIR, "artifacts.npz"),
    oof_cb=oof_cb, oof_xgb=oof_xgb, oof_lgb=oof_lgb, oof_hgb=oof_hgb,
    oof_et=oof_et, oof_knn=oof_knn, oof_chr=oof_chr,
    pred_cb=pred_cb, pred_xgb=pred_xgb, pred_lgb=pred_lgb, pred_hgb=pred_hgb,
    pred_et=pred_et, pred_knn=pred_knn, pred_chr=pred_chr,
    y_raw=y_raw, day49_idx=day49_idx,
    meta_coef=meta.coef_,
)
print(">> saved artifacts.npz (with Chronos)")



## v13 — two-stage residual model

Hard-codes the dominant signal `baseline = d48_same_ts` and trains a CatBoost on the cross-day delta `y - baseline`, using only day-49 train rows. Final pred = `baseline + delta`. This becomes the 8th stack member and the Ridge meta-learner refits over all 8 base models.

In [ ]:
"""
v13: Two-stage residual model. baseline = d48_same_ts. A CatBoost trained
ONLY on day-49 train rows learns the day-over-day delta = y - baseline.
Final pred = baseline + delta. Adds this as an 8th stack member and refits
the Ridge meta over 8 base models.
"""
import sys, io, os, warnings, time
warnings.filterwarnings("ignore")

class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, s):
        for st in self.streams:
            try: st.write(s); st.flush()
            except Exception: pass
    def flush(self):
        for st in self.streams:
            try: st.flush()
            except Exception: pass

DATA_DIR = os.path.dirname(os.path.abspath(__file__))
_logfile = open(os.path.join(DATA_DIR, "run.log"), "w", encoding="utf-8")
sys.stdout = _Tee(io.TextIOWrapper(sys.__stdout__.buffer, encoding="utf-8", line_buffering=True), _logfile)

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor

RNG = 42

print(">> loading data + v9 artifacts")
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

def parse_ts(s):
    h, m = s.str.split(":", expand=True).astype(int).T.values
    return h.astype(int), m.astype(int), (h * 60 + m).astype(int), (h * 4 + m // 15).astype(int)

train["hour"], train["minute"], train["tmin"], train["tslot"] = parse_ts(train["timestamp"])
test["hour"],  test["minute"],  test["tmin"],  test["tslot"]  = parse_ts(test["timestamp"])

# d48 lookup
d48 = train[train["day"] == 48][["geohash", "tslot", "demand"]]
d48_map = d48.set_index(["geohash", "tslot"])["demand"]

# per-gh d48 mean as fallback when (gh, tslot) missing from d48
gh_d48_mean = d48.groupby("geohash")["demand"].mean()
overall_d48_mean = float(d48["demand"].mean())

def fill_baseline(df):
    keys = list(zip(df["geohash"], df["tslot"]))
    base = pd.Series(keys, index=df.index).map(d48_map)
    base = base.fillna(df["geohash"].map(gh_d48_mean)).fillna(overall_d48_mean)
    return base.astype(np.float32).to_numpy()

train["baseline"] = fill_baseline(train)
test["baseline"]  = fill_baseline(test)
print(f"   train baseline NaN: {int(np.isnan(train['baseline']).sum())}")
print(f"   test baseline NaN:  {int(np.isnan(test['baseline']).sum())}")

# delta = y - baseline (only meaningful for day-49 train rows where day != baseline's day)
train["delta"] = train["demand"].astype(float) - train["baseline"]

# day-49 train rows = our training set for the delta model
d49 = train[train["day"] == 49].reset_index(drop=False).rename(columns={"index": "_orig_idx"})
print(f"   day-49 train rows: {len(d49)}")
print(f"   delta stats — mean: {d49['delta'].mean():.4f}, std: {d49['delta'].std():.4f}")


# --------------------------------------------------------------------------- #
# Build feature columns (subset that matters for cross-day delta)             #
# --------------------------------------------------------------------------- #
# Decode geohash → lat/lon (simple, used for the delta model)
GH = "0123456789bcdefghjkmnpqrstuvwxyz"
GH_MAP = {c: i for i, c in enumerate(GH)}
def decode(g):
    lat_lo, lat_hi = -90.0, 90.0
    lon_lo, lon_hi = -180.0, 180.0
    even = True
    for ch in g:
        bits = GH_MAP[ch]
        for m in (16, 8, 4, 2, 1):
            b = (bits & m) > 0
            if even:
                mid = (lon_lo + lon_hi) / 2
                if b: lon_lo = mid
                else: lon_hi = mid
            else:
                mid = (lat_lo + lat_hi) / 2
                if b: lat_lo = mid
                else: lat_hi = mid
            even = not even
    return (lat_lo + lat_hi)/2, (lon_lo + lon_hi)/2

all_gh = pd.unique(pd.concat([train["geohash"], test["geohash"]]))
gh_lat = {g: decode(g)[0] for g in all_gh}
gh_lon = {g: decode(g)[1] for g in all_gh}
train["lat"] = train["geohash"].map(gh_lat)
train["lon"] = train["geohash"].map(gh_lon)
test["lat"]  = test["geohash"].map(gh_lat)
test["lon"]  = test["geohash"].map(gh_lon)

for k in (3, 4, 5):
    train[f"gh{k}"] = train["geohash"].str.slice(0, k)
    test[f"gh{k}"]  = test["geohash"].str.slice(0, k)

# baseline-related features for the delta model
train["hour_sin"] = np.sin(2*np.pi*train["hour"]/24); train["hour_cos"] = np.cos(2*np.pi*train["hour"]/24)
test["hour_sin"]  = np.sin(2*np.pi*test["hour"]/24);  test["hour_cos"]  = np.cos(2*np.pi*test["hour"]/24)
train["tmin_sin"] = np.sin(2*np.pi*train["tmin"]/1440); train["tmin_cos"] = np.cos(2*np.pi*train["tmin"]/1440)
test["tmin_sin"]  = np.sin(2*np.pi*test["tmin"]/1440);  test["tmin_cos"]  = np.cos(2*np.pi*test["tmin"]/1440)

# d48 same-slot variability per gh (high std → unstable area → expect bigger delta)
gh_d48_stats = d48.groupby("geohash")["demand"].agg(["mean", "std", "max", "min"])
gh_d48_stats.columns = ["gh_d48_mean","gh_d48_std","gh_d48_max","gh_d48_min"]
train = train.merge(gh_d48_stats, left_on="geohash", right_index=True, how="left")
test  = test.merge(gh_d48_stats,  left_on="geohash", right_index=True, how="left")

CAT_FEATURES = ["geohash", "gh3", "gh4", "gh5", "RoadType", "LargeVehicles", "Landmarks", "Weather"]
NUM_FEATURES = ["hour", "minute", "tmin", "tslot", "hour_sin", "hour_cos", "tmin_sin", "tmin_cos",
                "lat", "lon", "NumberofLanes", "Temperature",
                "baseline", "gh_d48_mean", "gh_d48_std", "gh_d48_max", "gh_d48_min"]
FEATURES = CAT_FEATURES + NUM_FEATURES
for c in CAT_FEATURES:
    train[c] = train[c].astype("string").fillna("NA")
    test[c]  = test[c].astype("string").fillna("NA")


# --------------------------------------------------------------------------- #
# Train delta CatBoost on day-49 train rows with 5-fold CV                    #
# --------------------------------------------------------------------------- #
print(">> training 5-fold delta CatBoost on day-49 only (GPU)")
day49_idx = np.where(train["day"].values == 49)[0]
X_train = train[FEATURES].copy()
X_test = test[FEATURES].copy()
y_delta = train["delta"].astype(float).values

cb_params = dict(
    iterations=3000, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
    loss_function="RMSE", eval_metric="RMSE",
    random_seed=RNG, od_type="Iter", od_wait=200,
    verbose=300, task_type="GPU", devices="0",
)
cat_idx = [FEATURES.index(c) for c in CAT_FEATURES]

oof_delta = np.full(len(train), np.nan)
pred_delta = np.zeros(len(test))
kf = KFold(n_splits=5, shuffle=True, random_state=RNG)
for fold, (tr_d49, va_d49) in enumerate(kf.split(day49_idx), 1):
    tr_idx = day49_idx[tr_d49]
    va_idx = day49_idx[va_d49]
    model = CatBoostRegressor(**cb_params)
    model.fit(
        X_train.iloc[tr_idx], y_delta[tr_idx],
        eval_set=(X_train.iloc[va_idx], y_delta[va_idx]),
        cat_features=cat_idx, use_best_model=True,
    )
    oof_delta[va_idx] = model.predict(X_train.iloc[va_idx])
    pred_delta += model.predict(X_test) / kf.n_splits
    fold_r2 = r2_score(y_delta[va_idx], oof_delta[va_idx])
    print(f"  delta fold {fold}: best_iter={model.get_best_iteration()}  R²(delta)={fold_r2:.5f}")

print(f">> delta-target OOF R² on day-49 (delta scale, not raw): {r2_score(y_delta[day49_idx], oof_delta[day49_idx]):.5f}")

# Convert to demand-scale predictions: baseline + delta
oof_residual_model = train["baseline"].to_numpy() + oof_delta
pred_residual_model = test["baseline"].to_numpy() + pred_delta


# --------------------------------------------------------------------------- #
# Compare residual model to v9 base models, refit Ridge over 8 models         #
# --------------------------------------------------------------------------- #
art = np.load(os.path.join(DATA_DIR, "artifacts.npz"))
y_raw, day49_idx_art = art["y_raw"], art["day49_idx"]
assert np.array_equal(day49_idx, day49_idx_art), "day-49 index mismatch!"
y49 = y_raw[day49_idx]

resid_r2 = r2_score(y49, oof_residual_model[day49_idx])
print(f">> Two-stage residual model OOF R² (day-49, demand scale) = {resid_r2:.5f}  (score = {max(0,100*resid_r2):.3f})")

labels = ["cb", "xgb", "lgb", "hgb", "et", "knn", "chrS_50", "residual"]
oof_cols = [art["oof_cb"], art["oof_xgb"], art["oof_lgb"], art["oof_hgb"],
            art["oof_et"], art["oof_knn"], art["oof_chrS"], oof_residual_model]
test_cols = [art["pred_cb"], art["pred_xgb"], art["pred_lgb"], art["pred_hgb"],
             art["pred_et"], art["pred_knn"], art["pred_chrS"], pred_residual_model]

single_r2 = {lbl: r2_score(y49, c[day49_idx]) for lbl, c in zip(labels, oof_cols)}
print(">> single-model R²:", {k: round(v, 5) for k, v in single_r2.items()})

oof_stack = np.column_stack([c[day49_idx] for c in oof_cols])
test_stack = np.column_stack(test_cols)

meta = Ridge(alpha=0.1, fit_intercept=False, positive=True)
meta.fit(oof_stack, y49)
blend_r2 = r2_score(y49, meta.predict(oof_stack))
print(">> Ridge meta weights =", dict(zip(labels, np.round(meta.coef_, 4))))
print(f">> Ridge-stacked OOF R² (8 models) = {blend_r2:.5f}  (score = {max(0,100*blend_r2):.3f})")

pred = np.clip(meta.predict(test_stack), 0.0, 1.0)
sub = pd.DataFrame({"Index": test["Index"].values, "demand": pred})
sub.to_csv(os.path.join(DATA_DIR, "submission.csv"), index=False)
print(">> wrote submission.csv shape =", sub.shape)
print(sub.head())

np.savez(
    os.path.join(DATA_DIR, "artifacts.npz"),
    **{k: art[k] for k in art.files},
    oof_residual=oof_residual_model, pred_residual=pred_residual_model,
    meta_coef_v13=meta.coef_,
)
print(">> saved artifacts.npz")



## v15 — pseudo-labelled day-49 model

Uses v13 test predictions as pseudo-labels on the test rows, then trains a CatBoost on ~50k mixed `(real day-49 train + pseudo-labelled test)` examples — real rows get sample weight 1.0, pseudo rows 0.5. This lets the model see DAYTIME hours that my honest CV cannot, while still being validated only on real-labelled day-49 rows. Added as the 9th stack member; the Ridge meta refits.

In [ ]:
"""
v15: Pseudo-labeled CatBoost focused on day-49 patterns.
Combines day-49 train rows (real labels) with test rows (v13 predictions as
pseudo-labels) so the model trains on ~50k day-49 examples including the
DAYTIME hours my OOF cannot see. Adds as a 9th stack member, refits Ridge.
"""
import sys, io, os, warnings
warnings.filterwarnings("ignore")

class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, s):
        for st in self.streams:
            try: st.write(s); st.flush()
            except Exception: pass
    def flush(self):
        for st in self.streams:
            try: st.flush()
            except Exception: pass

DATA_DIR = os.path.dirname(os.path.abspath(__file__))
_logfile = open(os.path.join(DATA_DIR, "run.log"), "w", encoding="utf-8")
sys.stdout = _Tee(io.TextIOWrapper(sys.__stdout__.buffer, encoding="utf-8", line_buffering=True), _logfile)

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor

RNG = 42

print(">> loading data + artifacts + v13 submission as pseudo-labels")
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
v13_sub = pd.read_csv(os.path.join(DATA_DIR, "submission.csv"))  # current = v13 restored
test = test.merge(v13_sub.rename(columns={"demand": "pseudo_demand"}), on="Index", how="left")
print(f"   test rows with pseudo-label: {int(test['pseudo_demand'].notna().sum())} / {len(test)}")

# Build same features as v13 residual model
def parse_ts(s):
    h, m = s.str.split(":", expand=True).astype(int).T.values
    return h.astype(int), m.astype(int), (h * 60 + m).astype(int), (h * 4 + m // 15).astype(int)

train["hour"], train["minute"], train["tmin"], train["tslot"] = parse_ts(train["timestamp"])
test["hour"],  test["minute"],  test["tmin"],  test["tslot"]  = parse_ts(test["timestamp"])

# baseline = d48_same_ts (with fallback)
d48 = train[train["day"] == 48][["geohash", "tslot", "demand"]]
d48_map = d48.set_index(["geohash", "tslot"])["demand"]
gh_d48_mean = d48.groupby("geohash")["demand"].mean()
overall_d48_mean = float(d48["demand"].mean())

def fill_baseline(df):
    keys = list(zip(df["geohash"], df["tslot"]))
    base = pd.Series(keys, index=df.index).map(d48_map)
    base = base.fillna(df["geohash"].map(gh_d48_mean)).fillna(overall_d48_mean)
    return base.astype(np.float32).to_numpy()

train["baseline"] = fill_baseline(train)
test["baseline"]  = fill_baseline(test)

# decode geohash
GH = "0123456789bcdefghjkmnpqrstuvwxyz"
GH_MAP = {c: i for i, c in enumerate(GH)}
def decode(g):
    lat_lo, lat_hi, lon_lo, lon_hi = -90.0, 90.0, -180.0, 180.0
    even = True
    for ch in g:
        bits = GH_MAP[ch]
        for m in (16, 8, 4, 2, 1):
            b = (bits & m) > 0
            if even:
                mid = (lon_lo + lon_hi) / 2
                if b: lon_lo = mid
                else: lon_hi = mid
            else:
                mid = (lat_lo + lat_hi) / 2
                if b: lat_lo = mid
                else: lat_hi = mid
            even = not even
    return (lat_lo + lat_hi)/2, (lon_lo + lon_hi)/2

all_gh = pd.unique(pd.concat([train["geohash"], test["geohash"]]))
gh_lat = {g: decode(g)[0] for g in all_gh}
gh_lon = {g: decode(g)[1] for g in all_gh}
train["lat"] = train["geohash"].map(gh_lat); train["lon"] = train["geohash"].map(gh_lon)
test["lat"]  = test["geohash"].map(gh_lat);  test["lon"]  = test["geohash"].map(gh_lon)

for k in (3, 4, 5):
    train[f"gh{k}"] = train["geohash"].str.slice(0, k)
    test[f"gh{k}"]  = test["geohash"].str.slice(0, k)

train["hour_sin"] = np.sin(2*np.pi*train["hour"]/24); train["hour_cos"] = np.cos(2*np.pi*train["hour"]/24)
test["hour_sin"]  = np.sin(2*np.pi*test["hour"]/24);  test["hour_cos"]  = np.cos(2*np.pi*test["hour"]/24)
train["tmin_sin"] = np.sin(2*np.pi*train["tmin"]/1440); train["tmin_cos"] = np.cos(2*np.pi*train["tmin"]/1440)
test["tmin_sin"]  = np.sin(2*np.pi*test["tmin"]/1440);  test["tmin_cos"]  = np.cos(2*np.pi*test["tmin"]/1440)

gh_d48_stats = d48.groupby("geohash")["demand"].agg(["mean", "std", "max", "min"])
gh_d48_stats.columns = ["gh_d48_mean","gh_d48_std","gh_d48_max","gh_d48_min"]
train = train.merge(gh_d48_stats, left_on="geohash", right_index=True, how="left")
test  = test.merge(gh_d48_stats,  left_on="geohash", right_index=True, how="left")

CAT_FEATURES = ["geohash", "gh3", "gh4", "gh5", "RoadType", "LargeVehicles", "Landmarks", "Weather"]
NUM_FEATURES = ["day", "hour", "minute", "tmin", "tslot", "hour_sin", "hour_cos", "tmin_sin", "tmin_cos",
                "lat", "lon", "NumberofLanes", "Temperature",
                "baseline", "gh_d48_mean", "gh_d48_std", "gh_d48_max", "gh_d48_min"]
FEATURES = CAT_FEATURES + NUM_FEATURES
for c in CAT_FEATURES:
    train[c] = train[c].astype("string").fillna("NA")
    test[c]  = test[c].astype("string").fillna("NA")


# --------------------------------------------------------------------------- #
# Build the combined "day-49 + pseudo" training set                           #
# --------------------------------------------------------------------------- #
day49_idx = np.where(train["day"].values == 49)[0]
X_d49 = train.iloc[day49_idx][FEATURES].copy()
y_d49 = train.iloc[day49_idx]["demand"].astype(float).to_numpy()
X_test_full = test[FEATURES].copy()
# pseudo-set: day-49 (day=49 in feature set, since test is all day 49)
X_pseudo = test[FEATURES].copy()
y_pseudo = test["pseudo_demand"].astype(float).to_numpy()
print(f"   real day-49: {len(X_d49)}  pseudo (test): {len(X_pseudo)}  total: {len(X_d49)+len(X_pseudo)}")


# --------------------------------------------------------------------------- #
# 5-fold CV: hold out 20% of real day-49 as val, train on rest + all pseudo  #
# --------------------------------------------------------------------------- #
print(">> 5-fold pseudo-labeled CatBoost (real weight=1.0, pseudo weight=0.5, GPU)")
cb_params = dict(
    iterations=4000, learning_rate=0.05, depth=8, l2_leaf_reg=3.0,
    loss_function="RMSE", eval_metric="RMSE",
    random_seed=RNG, od_type="Iter", od_wait=200,
    verbose=500, task_type="GPU", devices="0",
)
cat_idx = [FEATURES.index(c) for c in CAT_FEATURES]

oof_pseudo = np.full(len(train), np.nan)
pred_pseudo = np.zeros(len(test))
kf = KFold(n_splits=5, shuffle=True, random_state=RNG)
PSEUDO_W = 0.5
for fold, (tr_d49_pos, va_d49_pos) in enumerate(kf.split(np.arange(len(X_d49))), 1):
    # train portion = real-train (80% of day-49) + ALL pseudo
    X_real_tr = X_d49.iloc[tr_d49_pos]
    y_real_tr = y_d49[tr_d49_pos]
    X_tr = pd.concat([X_real_tr, X_pseudo], ignore_index=True)
    y_tr = np.concatenate([y_real_tr, y_pseudo])
    w_tr = np.concatenate([np.ones(len(X_real_tr)), np.full(len(X_pseudo), PSEUDO_W)])

    X_va = X_d49.iloc[va_d49_pos]
    y_va = y_d49[va_d49_pos]
    va_real_idx = day49_idx[va_d49_pos]

    m = CatBoostRegressor(**cb_params)
    m.fit(X_tr, y_tr,
          sample_weight=w_tr,
          eval_set=(X_va, y_va),
          cat_features=cat_idx, use_best_model=True)

    oof_pseudo[va_real_idx] = m.predict(X_va)
    pred_pseudo += m.predict(X_test_full) / kf.n_splits
    fold_r2 = r2_score(y_va, oof_pseudo[va_real_idx])
    print(f"   fold {fold}: best_iter={m.get_best_iteration()}  R²(raw)={fold_r2:.5f}")

pseudo_oof_r2 = r2_score(y_d49, oof_pseudo[day49_idx])
print(f">> Pseudo-CatBoost OOF R² (day-49 raw) = {pseudo_oof_r2:.5f}  (score = {max(0,100*pseudo_oof_r2):.3f})")


# --------------------------------------------------------------------------- #
# Refit Ridge over 8 base models (v13's 8) + pseudo as 9th                    #
# --------------------------------------------------------------------------- #
art = np.load(os.path.join(DATA_DIR, "artifacts.npz"))
y_raw, day49_idx_art = art["y_raw"], art["day49_idx"]
y49 = y_raw[day49_idx]

labels = ["cb", "xgb", "lgb", "hgb", "et", "knn", "chrS_50", "residual", "pseudo"]
oof_cols = [art["oof_cb"], art["oof_xgb"], art["oof_lgb"], art["oof_hgb"],
            art["oof_et"], art["oof_knn"], art["oof_chrS"], art["oof_residual"], oof_pseudo]
test_cols = [art["pred_cb"], art["pred_xgb"], art["pred_lgb"], art["pred_hgb"],
             art["pred_et"], art["pred_knn"], art["pred_chrS"], art["pred_residual"], pred_pseudo]

single_r2 = {lbl: r2_score(y49, c[day49_idx]) for lbl, c in zip(labels, oof_cols)}
print(">> single-model R²:", {k: round(v, 5) for k, v in single_r2.items()})

oof_stack = np.column_stack([c[day49_idx] for c in oof_cols])
test_stack = np.column_stack(test_cols)

meta = Ridge(alpha=0.1, fit_intercept=False, positive=True)
meta.fit(oof_stack, y49)
blend_r2 = r2_score(y49, meta.predict(oof_stack))
print(">> Ridge meta weights =", dict(zip(labels, np.round(meta.coef_, 4))))
print(f">> Ridge-stacked OOF R² (9 models) = {blend_r2:.5f}  (score = {max(0,100*blend_r2):.3f})")

pred = np.clip(meta.predict(test_stack), 0.0, 1.0)
sub = pd.DataFrame({"Index": test["Index"].values, "demand": pred})
sub.to_csv(os.path.join(DATA_DIR, "submission.csv"), index=False)
print(">> wrote submission.csv shape =", sub.shape)
print(sub.head())

np.savez(
    os.path.join(DATA_DIR, "artifacts.npz"),
    **{k: art[k] for k in art.files},
    oof_pseudo=oof_pseudo, pred_pseudo=pred_pseudo,
    meta_coef_v15=meta.coef_,
)
print(">> saved artifacts.npz")



## v16 — PyTorch NN with categorical embeddings

A custom MLP with embeddings for `geohash`, `hour`, and other categoricals + numeric features. Trained with the same 5-fold day-49 honest CV and sample weighting as the GBDT base models. Added as the 10th stack member. The Ridge meta-learner assigned it weight 0, indicating its predictions overlap too much with the trees to contribute uniquely — but it's documented here as a completeness check on alternative architectures.

In [ ]:
"""
v16: Custom PyTorch NN with categorical embeddings + MLP head.
Genuinely different inductive bias from trees and Chronos.
Trained with the same 5-fold day-49 honest CV as the other base models;
added to the Ridge stack as the 10th member.
"""
import sys, io, os, warnings, time
warnings.filterwarnings("ignore")

class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, s):
        for st in self.streams:
            try: st.write(s); st.flush()
            except Exception: pass
    def flush(self):
        for st in self.streams:
            try: st.flush()
            except Exception: pass

DATA_DIR = os.path.dirname(os.path.abspath(__file__))
_logfile = open(os.path.join(DATA_DIR, "run.log"), "w", encoding="utf-8")
sys.stdout = _Tee(io.TextIOWrapper(sys.__stdout__.buffer, encoding="utf-8", line_buffering=True), _logfile)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

RNG = 42
torch.manual_seed(RNG); np.random.seed(RNG)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f">> torch device: {DEVICE}")


# --------------------------------------------------------------------------- #
# Load data + minimal features for NN                                         #
# --------------------------------------------------------------------------- #
print(">> loading data")
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

def parse_ts(s):
    h, m = s.str.split(":", expand=True).astype(int).T.values
    return h.astype(int), m.astype(int), (h * 60 + m).astype(int), (h * 4 + m // 15).astype(int)

train["hour"], train["minute"], train["tmin"], train["tslot"] = parse_ts(train["timestamp"])
test["hour"],  test["minute"],  test["tmin"],  test["tslot"]  = parse_ts(test["timestamp"])

# d48 lookup
d48 = train[train["day"] == 48][["geohash", "tslot", "demand"]]
d48_map = d48.set_index(["geohash", "tslot"])["demand"]
gh_d48_mean = d48.groupby("geohash")["demand"].mean()
gh_d48_std  = d48.groupby("geohash")["demand"].std()
overall_d48_mean = float(d48["demand"].mean())

def fill_baseline(df):
    keys = list(zip(df["geohash"], df["tslot"]))
    return pd.Series(keys, index=df.index).map(d48_map)\
            .fillna(df["geohash"].map(gh_d48_mean)).fillna(overall_d48_mean)\
            .astype(np.float32).to_numpy()

train["d48_same_ts"] = fill_baseline(train)
test["d48_same_ts"]  = fill_baseline(test)
train["gh_d48_mean"] = train["geohash"].map(gh_d48_mean).fillna(overall_d48_mean).astype(np.float32)
test["gh_d48_mean"]  = test["geohash"].map(gh_d48_mean).fillna(overall_d48_mean).astype(np.float32)
train["gh_d48_std"]  = train["geohash"].map(gh_d48_std).fillna(0.0).astype(np.float32)
test["gh_d48_std"]   = test["geohash"].map(gh_d48_std).fillna(0.0).astype(np.float32)

# gh × hour mean
gh_hour = d48.copy()
gh_hour["hour"] = gh_hour["tslot"] // 4
gh_hour_mean = gh_hour.groupby(["geohash", "hour"])["demand"].mean()
def gh_hour_lookup(df):
    return df.set_index(["geohash", "hour"]).index.map(gh_hour_mean).to_series(index=df.index).fillna(overall_d48_mean).astype(np.float32).to_numpy()
train["gh_hour_d48_mean"] = gh_hour_lookup(train)
test["gh_hour_d48_mean"]  = gh_hour_lookup(test)

# d49 recent (max-tslot day-49 demand for this geohash with tslot < query)
d49 = train[train["day"] == 49][["geohash", "tslot", "demand"]].rename(columns={"demand": "d49_recent"}).sort_values("tslot").reset_index(drop=True)
def attach_d49_recent(df):
    src = df[["geohash", "tslot"]].copy()
    src["_orig"] = np.arange(len(src))
    src = src.sort_values("tslot")
    merged = pd.merge_asof(src, d49, on="tslot", by="geohash", direction="backward", allow_exact_matches=False)
    merged = merged.sort_values("_orig")
    return merged["d49_recent"].to_numpy()
train["d49_recent"] = attach_d49_recent(train)
test["d49_recent"]  = attach_d49_recent(test)

# decode geohash
GH = "0123456789bcdefghjkmnpqrstuvwxyz"
GH_MAP = {c: i for i, c in enumerate(GH)}
def decode(g):
    lat_lo, lat_hi, lon_lo, lon_hi = -90.0, 90.0, -180.0, 180.0
    even = True
    for ch in g:
        bits = GH_MAP[ch]
        for m in (16, 8, 4, 2, 1):
            b = (bits & m) > 0
            if even:
                mid = (lon_lo + lon_hi)/2
                if b: lon_lo = mid
                else: lon_hi = mid
            else:
                mid = (lat_lo + lat_hi)/2
                if b: lat_lo = mid
                else: lat_hi = mid
            even = not even
    return (lat_lo + lat_hi)/2, (lon_lo + lon_hi)/2

all_gh = pd.unique(pd.concat([train["geohash"], test["geohash"]]))
gh_lat = {g: decode(g)[0] for g in all_gh}
gh_lon = {g: decode(g)[1] for g in all_gh}
train["lat"] = train["geohash"].map(gh_lat); train["lon"] = train["geohash"].map(gh_lon)
test["lat"]  = test["geohash"].map(gh_lat);  test["lon"]  = test["geohash"].map(gh_lon)

train["hour_sin"] = np.sin(2*np.pi*train["hour"]/24); train["hour_cos"] = np.cos(2*np.pi*train["hour"]/24)
test["hour_sin"]  = np.sin(2*np.pi*test["hour"]/24);  test["hour_cos"]  = np.cos(2*np.pi*test["hour"]/24)


# --------------------------------------------------------------------------- #
# Build NN inputs                                                             #
# --------------------------------------------------------------------------- #
CAT_COLS = ["geohash", "RoadType", "LargeVehicles", "Landmarks", "Weather", "hour"]
NUM_COLS = ["day", "tslot", "tmin", "hour_sin", "hour_cos",
            "lat", "lon", "NumberofLanes", "Temperature",
            "d48_same_ts", "gh_d48_mean", "gh_d48_std",
            "gh_hour_d48_mean", "d49_recent"]

# build vocabularies
cat_vocab = {}
for c in CAT_COLS:
    vals = pd.concat([train[c].astype("string").fillna("NA"),
                      test[c].astype("string").fillna("NA")], ignore_index=True)
    cat_vocab[c] = {v: i + 1 for i, v in enumerate(vals.unique())}  # +1 so 0 = padding/unknown
    cat_vocab[c]["__UNK__"] = 0
    print(f"   cat {c}: {len(cat_vocab[c])} unique")

def encode_cat(df):
    out = np.zeros((len(df), len(CAT_COLS)), dtype=np.int64)
    for i, c in enumerate(CAT_COLS):
        out[:, i] = df[c].astype("string").fillna("NA").map(cat_vocab[c]).fillna(0).astype(np.int64).to_numpy()
    return out

X_cat_train = encode_cat(train)
X_cat_test  = encode_cat(test)

# numeric: impute median + standardize
imp = SimpleImputer(strategy="median")
sc = StandardScaler()
X_num_train = sc.fit_transform(imp.fit_transform(train[NUM_COLS])).astype(np.float32)
X_num_test  = sc.transform(imp.transform(test[NUM_COLS])).astype(np.float32)
y_train = train["demand"].astype(np.float32).to_numpy()
print(f"   num_features: {X_num_train.shape[1]}, cat_features: {X_cat_train.shape[1]}")


# --------------------------------------------------------------------------- #
# Model                                                                       #
# --------------------------------------------------------------------------- #
class TabNN(nn.Module):
    def __init__(self, cat_sizes, num_dim, hidden=(256, 128, 64), dropout=0.2):
        super().__init__()
        self.embeds = nn.ModuleList([nn.Embedding(sz, min(32, max(4, sz // 8))) for sz in cat_sizes])
        emb_dim = sum(e.embedding_dim for e in self.embeds)
        in_dim = emb_dim + num_dim
        layers = []
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers += [nn.Linear(in_dim, 1)]
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_cat, x_num):
        embs = [e(x_cat[:, i]) for i, e in enumerate(self.embeds)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.mlp(x).squeeze(-1)


# --------------------------------------------------------------------------- #
# 5-fold CV on day-49 honest val                                              #
# --------------------------------------------------------------------------- #
day49_idx = np.where(train["day"].values == 49)[0]
day48_idx = np.where(train["day"].values == 48)[0]

# sample weights matching the GBDT pipeline (day-48 daytime: 1.5, day-49: 5.0)
sample_weight = np.ones(len(train), dtype=np.float32)
sample_weight[(train["day"].values == 48) & (train["tslot"].values >= 9)] = 1.5
sample_weight[train["day"].values == 49] = 5.0

cat_sizes = [len(cat_vocab[c]) + 1 for c in CAT_COLS]  # +1 safety for max idx
print(f"   cat embedding sizes: {cat_sizes}")

oof_nn = np.full(len(train), np.nan, dtype=np.float32)
pred_nn = np.zeros(len(test), dtype=np.float32)
kf = KFold(n_splits=5, shuffle=True, random_state=RNG)

BATCH = 4096
EPOCHS = 80
PATIENCE = 12

for fold, (tr_d49, va_d49) in enumerate(kf.split(day49_idx), 1):
    tr_idx = np.concatenate([day48_idx, day49_idx[tr_d49]])
    va_idx = day49_idx[va_d49]

    Xc_tr = torch.tensor(X_cat_train[tr_idx]); Xn_tr = torch.tensor(X_num_train[tr_idx])
    y_tr  = torch.tensor(y_train[tr_idx]); w_tr = torch.tensor(sample_weight[tr_idx])
    Xc_va = torch.tensor(X_cat_train[va_idx]).to(DEVICE)
    Xn_va = torch.tensor(X_num_train[va_idx]).to(DEVICE)
    y_va_np = y_train[va_idx]

    ds = TensorDataset(Xc_tr, Xn_tr, y_tr, w_tr)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=False, num_workers=0, pin_memory=True)

    model = TabNN(cat_sizes, X_num_train.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    best_r2 = -1; best_pred = None; bad = 0
    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum = 0.0; n = 0
        for xc, xn, y, w in loader:
            xc = xc.to(DEVICE, non_blocking=True); xn = xn.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True); w = w.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            pred = model(xc, xn)
            loss = ((pred - y) ** 2 * w).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
            loss_sum += loss.item() * y.size(0); n += y.size(0)
        sched.step()

        model.eval()
        with torch.no_grad():
            v_pred = model(Xc_va, Xn_va).cpu().numpy()
        v_r2 = r2_score(y_va_np, v_pred)
        if v_r2 > best_r2 + 1e-5:
            best_r2 = v_r2
            best_pred = v_pred
            # also stash test pred at this checkpoint
            with torch.no_grad():
                t_chunks = []
                for i in range(0, len(X_cat_test), 8192):
                    Xc_t = torch.tensor(X_cat_test[i:i+8192]).to(DEVICE)
                    Xn_t = torch.tensor(X_num_test[i:i+8192]).to(DEVICE)
                    t_chunks.append(model(Xc_t, Xn_t).cpu().numpy())
                best_test_pred = np.concatenate(t_chunks)
            bad = 0
        else:
            bad += 1
        if epoch % 10 == 0 or epoch == EPOCHS - 1:
            print(f"   fold {fold} epoch {epoch:3d}  train_loss={loss_sum/n:.6f}  val_R²={v_r2:.5f}  best={best_r2:.5f}")
        if bad >= PATIENCE:
            print(f"   fold {fold} early stop at epoch {epoch}, best val R²={best_r2:.5f}")
            break

    oof_nn[va_idx] = best_pred
    pred_nn += best_test_pred / kf.n_splits
    print(f"  >> fold {fold}: val R²={best_r2:.5f}  time={time.time()-t0:.1f}s")

nn_oof_r2 = r2_score(y_train[day49_idx], oof_nn[day49_idx])
print(f">> NN OOF R² (day-49 raw) = {nn_oof_r2:.5f}  (score = {max(0,100*nn_oof_r2):.3f})")


# --------------------------------------------------------------------------- #
# Refit Ridge over base models + NN                                           #
# --------------------------------------------------------------------------- #
art = np.load(os.path.join(DATA_DIR, "artifacts.npz"))
y_raw, day49_idx_art = art["y_raw"], art["day49_idx"]
y49 = y_raw[day49_idx]

labels = ["cb", "xgb", "lgb", "hgb", "et", "knn", "chrS_50", "residual", "pseudo", "nn"]
oof_cols = [art["oof_cb"], art["oof_xgb"], art["oof_lgb"], art["oof_hgb"],
            art["oof_et"], art["oof_knn"], art["oof_chrS"],
            art["oof_residual"], art["oof_pseudo"], oof_nn]
test_cols = [art["pred_cb"], art["pred_xgb"], art["pred_lgb"], art["pred_hgb"],
             art["pred_et"], art["pred_knn"], art["pred_chrS"],
             art["pred_residual"], art["pred_pseudo"], pred_nn]

single_r2 = {lbl: r2_score(y49, c[day49_idx]) for lbl, c in zip(labels, oof_cols)}
print(">> single-model R²:", {k: round(v, 5) for k, v in single_r2.items()})

oof_stack = np.column_stack([c[day49_idx] for c in oof_cols])
test_stack = np.column_stack(test_cols)

meta = Ridge(alpha=0.1, fit_intercept=False, positive=True)
meta.fit(oof_stack, y49)
blend_r2 = r2_score(y49, meta.predict(oof_stack))
print(">> Ridge meta weights =", dict(zip(labels, np.round(meta.coef_, 4))))
print(f">> Ridge-stacked OOF R² (10 models) = {blend_r2:.5f}  (score = {max(0,100*blend_r2):.3f})")

pred = np.clip(meta.predict(test_stack), 0.0, 1.0)
sub = pd.DataFrame({"Index": test["Index"].values, "demand": pred})
sub.to_csv(os.path.join(DATA_DIR, "submission.csv"), index=False)
print(">> wrote submission.csv shape =", sub.shape)
print(sub.head())

np.savez(
    os.path.join(DATA_DIR, "artifacts.npz"),
    **{k: art[k] for k in art.files},
    oof_nn=oof_nn, pred_nn=pred_nn,
    meta_coef_v16=meta.coef_,
)
print(">> saved artifacts.npz")



## v17 — pseudo-labelled XGBoost + LightGBM

Extends v15's pseudo-labeling to two more model families. Trains separate XGBoost and LightGBM models on `(day-49 train + pseudo-labelled test)` with real-row weight 1.0 and pseudo-row weight 0.5. Both gave the Ridge meta-learner the highest single-model weights it has ever assigned (pseudo-XGB 0.17, pseudo-LGB 0.155), lifting stacked OOF from 0.9643 to 0.9653.

In [ ]:
"""
v17: Pseudo-labeled XGBoost + LightGBM (different inductive bias from v15's CatBoost).
Uses v15 submission predictions as pseudo-labels on test rows, trains both XGBoost
and LightGBM on (real day-49 + pseudo-labeled test) with weight 1.0 for real, 0.5 for
pseudo. Adds both as stack members; Ridge refits over 12 base models.
"""
import sys, io, os, warnings, time
warnings.filterwarnings("ignore")

class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, s):
        for st in self.streams:
            try: st.write(s); st.flush()
            except Exception: pass
    def flush(self):
        for st in self.streams:
            try: st.flush()
            except Exception: pass

DATA_DIR = os.path.dirname(os.path.abspath(__file__))
_logfile = open(os.path.join(DATA_DIR, "run.log"), "w", encoding="utf-8")
sys.stdout = _Tee(io.TextIOWrapper(sys.__stdout__.buffer, encoding="utf-8", line_buffering=True), _logfile)

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
import xgboost as xgb
import lightgbm as lgb

RNG = 42

print(">> loading data + current submission as pseudo-labels")
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sub_now = pd.read_csv(os.path.join(DATA_DIR, "submission.csv"))
test = test.merge(sub_now.rename(columns={"demand": "pseudo_demand"}), on="Index", how="left")
print(f"   pseudo-labels available: {int(test['pseudo_demand'].notna().sum())} / {len(test)}")


# --------------------------------------------------------------------------- #
# Feature engineering — same minimal subset as v15                            #
# --------------------------------------------------------------------------- #
def parse_ts(s):
    h, m = s.str.split(":", expand=True).astype(int).T.values
    return h.astype(int), m.astype(int), (h * 60 + m).astype(int), (h * 4 + m // 15).astype(int)

train["hour"], train["minute"], train["tmin"], train["tslot"] = parse_ts(train["timestamp"])
test["hour"],  test["minute"],  test["tmin"],  test["tslot"]  = parse_ts(test["timestamp"])

d48 = train[train["day"] == 48][["geohash", "tslot", "demand"]]
d48_map = d48.set_index(["geohash", "tslot"])["demand"]
gh_d48_mean = d48.groupby("geohash")["demand"].mean()
gh_d48_std  = d48.groupby("geohash")["demand"].std()
overall_d48_mean = float(d48["demand"].mean())

def fill_baseline(df):
    keys = list(zip(df["geohash"], df["tslot"]))
    return pd.Series(keys, index=df.index).map(d48_map)\
            .fillna(df["geohash"].map(gh_d48_mean)).fillna(overall_d48_mean)\
            .astype(np.float32).to_numpy()

train["baseline"] = fill_baseline(train)
test["baseline"]  = fill_baseline(test)
train["gh_d48_mean"] = train["geohash"].map(gh_d48_mean).fillna(overall_d48_mean).astype(np.float32)
test["gh_d48_mean"]  = test["geohash"].map(gh_d48_mean).fillna(overall_d48_mean).astype(np.float32)
train["gh_d48_std"]  = train["geohash"].map(gh_d48_std).fillna(0.0).astype(np.float32)
test["gh_d48_std"]   = test["geohash"].map(gh_d48_std).fillna(0.0).astype(np.float32)

GH = "0123456789bcdefghjkmnpqrstuvwxyz"
GH_MAP = {c: i for i, c in enumerate(GH)}
def decode(g):
    lat_lo, lat_hi, lon_lo, lon_hi = -90.0, 90.0, -180.0, 180.0
    even = True
    for ch in g:
        bits = GH_MAP[ch]
        for m in (16, 8, 4, 2, 1):
            b = (bits & m) > 0
            if even:
                mid = (lon_lo + lon_hi)/2
                if b: lon_lo = mid
                else: lon_hi = mid
            else:
                mid = (lat_lo + lat_hi)/2
                if b: lat_lo = mid
                else: lat_hi = mid
            even = not even
    return (lat_lo + lat_hi)/2, (lon_lo + lon_hi)/2

all_gh = pd.unique(pd.concat([train["geohash"], test["geohash"]]))
gh_lat = {g: decode(g)[0] for g in all_gh}
gh_lon = {g: decode(g)[1] for g in all_gh}
train["lat"] = train["geohash"].map(gh_lat); train["lon"] = train["geohash"].map(gh_lon)
test["lat"]  = test["geohash"].map(gh_lat);  test["lon"]  = test["geohash"].map(gh_lon)

train["hour_sin"] = np.sin(2*np.pi*train["hour"]/24); train["hour_cos"] = np.cos(2*np.pi*train["hour"]/24)
test["hour_sin"]  = np.sin(2*np.pi*test["hour"]/24);  test["hour_cos"]  = np.cos(2*np.pi*test["hour"]/24)

for k in (3, 4, 5):
    train[f"gh{k}"] = train["geohash"].str.slice(0, k)
    test[f"gh{k}"]  = test["geohash"].str.slice(0, k)


# label-encode categoricals for XGBoost/LightGBM
CAT_FEATURES = ["geohash", "gh3", "gh4", "gh5", "RoadType", "LargeVehicles", "Landmarks", "Weather"]
NUM_FEATURES = ["day", "hour", "minute", "tmin", "tslot", "hour_sin", "hour_cos",
                "lat", "lon", "NumberofLanes", "Temperature",
                "baseline", "gh_d48_mean", "gh_d48_std"]
FEATURES = CAT_FEATURES + NUM_FEATURES

X_full = pd.concat([train[FEATURES].copy(), test[FEATURES].copy()], ignore_index=True)
for c in CAT_FEATURES:
    X_full[c] = X_full[c].astype("string").fillna("NA")
    codes, _ = pd.factorize(X_full[c], sort=True)
    X_full[c] = codes
X_full = X_full.astype(np.float32)
X_train = X_full.iloc[:len(train)].reset_index(drop=True)
X_test  = X_full.iloc[len(train):].reset_index(drop=True)
y_train = train["demand"].astype(float).to_numpy()
y_pseudo = test["pseudo_demand"].astype(float).to_numpy()


# --------------------------------------------------------------------------- #
# 5-fold CV for pseudo-labeled XGBoost                                        #
# --------------------------------------------------------------------------- #
day49_idx = np.where(train["day"].values == 49)[0]
X_d49 = X_train.iloc[day49_idx].reset_index(drop=True)
y_d49 = y_train[day49_idx]
print(f"   day-49 real rows: {len(X_d49)}  pseudo test rows: {len(X_test)}")

PSEUDO_W = 0.5
kf = KFold(n_splits=5, shuffle=True, random_state=RNG)

print(">> pseudo-labeled XGBoost (5-fold, GPU)")
oof_pseudo_xgb = np.full(len(train), np.nan)
pred_pseudo_xgb = np.zeros(len(test))
xgb_params = dict(
    n_estimators=4000, learning_rate=0.05, max_depth=8,
    subsample=0.85, colsample_bytree=0.85, min_child_weight=4, reg_lambda=1.0,
    tree_method="hist", device="cuda", objective="reg:squarederror",
    random_state=RNG, n_jobs=-1, early_stopping_rounds=200,
)
for fold, (tr_pos, va_pos) in enumerate(kf.split(np.arange(len(X_d49))), 1):
    X_real_tr = X_d49.iloc[tr_pos]; y_real_tr = y_d49[tr_pos]
    X_tr = pd.concat([X_real_tr, X_test], ignore_index=True)
    y_tr = np.concatenate([y_real_tr, y_pseudo])
    w_tr = np.concatenate([np.ones(len(X_real_tr)), np.full(len(X_test), PSEUDO_W)])
    X_va = X_d49.iloc[va_pos]; y_va = y_d49[va_pos]
    va_real_idx = day49_idx[va_pos]

    m = xgb.XGBRegressor(**xgb_params)
    m.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_pseudo_xgb[va_real_idx] = m.predict(X_va)
    pred_pseudo_xgb += m.predict(X_test) / kf.n_splits
    print(f"   xgb fold {fold}: best_iter={m.best_iteration}  R²={r2_score(y_va, oof_pseudo_xgb[va_real_idx]):.5f}")

xgb_pseudo_r2 = r2_score(y_d49, oof_pseudo_xgb[day49_idx])
print(f">> Pseudo-XGBoost OOF R² (day-49 raw) = {xgb_pseudo_r2:.5f}  (score = {max(0,100*xgb_pseudo_r2):.3f})")


print(">> pseudo-labeled LightGBM (5-fold, GPU)")
oof_pseudo_lgb = np.full(len(train), np.nan)
pred_pseudo_lgb = np.zeros(len(test))
lgb_params = dict(
    n_estimators=4000, learning_rate=0.05, num_leaves=127,
    min_child_samples=20, subsample=0.85, colsample_bytree=0.85,
    reg_lambda=1.0, device="gpu", objective="regression", metric="rmse",
    random_state=RNG, n_jobs=-1, verbose=-1,
)
for fold, (tr_pos, va_pos) in enumerate(kf.split(np.arange(len(X_d49))), 1):
    X_real_tr = X_d49.iloc[tr_pos]; y_real_tr = y_d49[tr_pos]
    X_tr = pd.concat([X_real_tr, X_test], ignore_index=True)
    y_tr = np.concatenate([y_real_tr, y_pseudo])
    w_tr = np.concatenate([np.ones(len(X_real_tr)), np.full(len(X_test), PSEUDO_W)])
    X_va = X_d49.iloc[va_pos]; y_va = y_d49[va_pos]
    va_real_idx = day49_idx[va_pos]

    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(X_tr, y_tr, sample_weight=w_tr,
          eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(200, verbose=False)])
    oof_pseudo_lgb[va_real_idx] = m.predict(X_va)
    pred_pseudo_lgb += m.predict(X_test) / kf.n_splits
    print(f"   lgb fold {fold}: best_iter={m.best_iteration_}  R²={r2_score(y_va, oof_pseudo_lgb[va_real_idx]):.5f}")

lgb_pseudo_r2 = r2_score(y_d49, oof_pseudo_lgb[day49_idx])
print(f">> Pseudo-LightGBM OOF R² (day-49 raw) = {lgb_pseudo_r2:.5f}  (score = {max(0,100*lgb_pseudo_r2):.3f})")


# --------------------------------------------------------------------------- #
# Refit Ridge stack with 12 base models                                       #
# --------------------------------------------------------------------------- #
art = np.load(os.path.join(DATA_DIR, "artifacts.npz"))
y_raw = art["y_raw"]; day49_idx_art = art["day49_idx"]
y49 = y_raw[day49_idx]

labels = ["cb","xgb","lgb","hgb","et","knn","chrS_50","residual","pseudo","nn","pseudoXGB","pseudoLGB"]
oof_cols = [art["oof_cb"], art["oof_xgb"], art["oof_lgb"], art["oof_hgb"],
            art["oof_et"], art["oof_knn"], art["oof_chrS"],
            art["oof_residual"], art["oof_pseudo"], art["oof_nn"],
            oof_pseudo_xgb, oof_pseudo_lgb]
test_cols = [art["pred_cb"], art["pred_xgb"], art["pred_lgb"], art["pred_hgb"],
             art["pred_et"], art["pred_knn"], art["pred_chrS"],
             art["pred_residual"], art["pred_pseudo"], art["pred_nn"],
             pred_pseudo_xgb, pred_pseudo_lgb]

single_r2 = {lbl: r2_score(y49, c[day49_idx]) for lbl, c in zip(labels, oof_cols)}
print(">> single-model R²:", {k: round(v, 5) for k, v in single_r2.items()})

oof_stack = np.column_stack([c[day49_idx] for c in oof_cols])
test_stack = np.column_stack(test_cols)

meta = Ridge(alpha=0.1, fit_intercept=False, positive=True)
meta.fit(oof_stack, y49)
blend_r2 = r2_score(y49, meta.predict(oof_stack))
print(">> Ridge meta weights =", dict(zip(labels, np.round(meta.coef_, 4))))
print(f">> Ridge-stacked OOF R² (12 models) = {blend_r2:.5f}  (score = {max(0,100*blend_r2):.3f})")

pred = np.clip(meta.predict(test_stack), 0.0, 1.0)
sub = pd.DataFrame({"Index": test["Index"].values, "demand": pred})
sub.to_csv(os.path.join(DATA_DIR, "submission.csv"), index=False)
print(">> wrote submission.csv shape =", sub.shape)
print(sub.head())

np.savez(
    os.path.join(DATA_DIR, "artifacts.npz"),
    **{k: art[k] for k in art.files},
    oof_pseudo_xgb=oof_pseudo_xgb, pred_pseudo_xgb=pred_pseudo_xgb,
    oof_pseudo_lgb=oof_pseudo_lgb, pred_pseudo_lgb=pred_pseudo_lgb,
    meta_coef_v17=meta.coef_,
)
print(">> saved artifacts.npz")

